# Gaussian Bump Array Generator (Nazca)

**Author:** Jason P. Beech  
**Date:** 2026-03  
**Affiliation:** Tegenfeldt Lab / Lund University  

This notebook generates an `N × N` array of circular shapes with a localized Gaussian bump in **Nazca**.

## What the code does
- Each object starts from a base circle of radius `R_um`.
- A Gaussian-shaped radial bump is added at angle `theta0_deg`.
- The bump amplitude varies uniformly across the array, from `A_max_um` at the top-left to `A_min_um` at the bottom-right.
- The bump width is set by `sigma_deg`, which in this notebook is scaled proportionally to `|A|`.

## Main settings
At the top of the code cell, you can change:

- `EXPORT_MODE`
  - `"plot"` → preview only
  - `"gds"` → export GDS only
- `N` → array size (`N × N`)
- `R_um` → base radius
- `A_max_um`, `A_min_um` → amplitude range across the array
- `theta0_deg` → angular position of the bump
- `sigma_scale` → scaling factor that sets the bump width from amplitude
- `pitchX_um`, `pitchY_um` → center-to-center spacing
- `snake` → optional snake ordering for amplitude assignment

## Geometry notes
- The radial profile is:

  \[
  r(\theta) = R + A \exp\left(-\frac{1}{2}\left(\frac{\Delta \theta}{\sigma}\right)^2\right)
  \]

- The array is centered about the origin.
- Amplitudes are assigned in row-major order from top-left to bottom-right.
- If `snake = True`, every other row is reversed while keeping the amplitude spacing uniform.

## Output behavior
- `"plot"` → shows a Nazca preview
- `"gds"` → exports a GDSII file

## Automatic filename
The exported filename is generated automatically from the main design parameters and the current date.

It includes:

- array size: `N x N`
- base radius: `R_um`
- amplitude range: `A_max_um` to `A_min_um`
- sigma scale: `sigma_scale`
- date in `YYYYMMDD` format

Example:

```python
gaussian_bump_array_4x4_R50um_A49to-49_sigmaScale0.2_20260317.gds

In [4]:
import nazca as nd
import math
from datetime import datetime

# =============================================================================
# Gaussian Bump Array Generator (Nazca)
#
# Author: Jason P. Beech
# Date: 2026-03
# Affiliation: Tegenfeldt Lab / Lund University
#
# Description:
# Generates an N x N array of circular shapes with a localized Gaussian bump.
# The bump amplitude varies uniformly across the array.
# =============================================================================


# =============================================================================
# Output control
# =============================================================================
# Choose one of:
#   "plot" -> preview only
#   "gds"  -> export GDS only
EXPORT_MODE = "gds"


# =============================================================================
# Design parameters (units: µm unless otherwise noted)
# =============================================================================

N = 4                 # array size (N x N)
R_um = 50.0           # base radius
A_max_um = +49.0      # amplitude at top-left
A_min_um = -49.0      # amplitude at bottom-right
theta0_deg = 45.0     # bump angle on the circle (0 = +x, 90 = +y)
sigma_deg = R_um      # initial placeholder; overwritten below from sigma_scale
N_samp = 720          # polygon resolution around circle
layer = 1

sigma_scale = 0.2     # degrees per micron of |A|

# Center-to-center spacing
pitchX_um = 3.0 * R_um
pitchY_um = 3.0 * R_um

# Optional boustrophedon (snake) sweep row-by-row while preserving uniform amplitudes
snake = False


# =============================================================================
# Auto-generated filename
# =============================================================================

date_tag = datetime.now().strftime("%Y%m%d")

GDS_FILENAME = (
    f"gaussian_bump_array_"
    f"{N}x{N}_"
    f"R{R_um:g}um_"
    f"A{A_max_um:g}to{A_min_um:g}_"
    f"sigmaScale{sigma_scale:g}_"
    f"{date_tag}.gds"
)


# =============================================================================
# Helper functions
# =============================================================================

def angwrap(a):
    """
    Wrap an angle to the interval [-pi, pi].
    """
    return math.atan2(math.sin(a), math.cos(a))


def circle_gauss_bump(xc, yc, R, A, theta0_deg=0.0, sigma_deg=10.0, n=720, layer=1):
    """
    Draw a circle with a localized Gaussian bump in polar radius.

    Parameters
    ----------
    xc, yc : float
        Center coordinates.
    R : float
        Base radius in µm.
    A : float
        Bump amplitude in µm.
    theta0_deg : float
        Angular position of the bump in degrees.
    sigma_deg : float
        Angular standard deviation of the bump in degrees.
    n : int
        Number of polygon samples around the shape.
    layer : int
        GDS layer number.
    """
    t0 = math.radians(theta0_deg)
    s = math.radians(sigma_deg)

    theta = [2 * math.pi * i / n for i in range(n)]
    r = [R + A * math.exp(-0.5 * (angwrap(t - t0) / s) ** 2) for t in theta]

    if min(r) <= 0.0:
        raise ValueError(
            f"Non-positive radius (min r = {min(r):.3f} µm). "
            "Reduce |A| or increase R/sigma."
        )

    pts = [(xc + r[i] * math.cos(theta[i]), yc + r[i] * math.sin(theta[i])) for i in range(n)]
    pts.append(pts[0])  # close polygon

    nd.Polygon(pts, layer=layer).put(0, 0)


def amp_uniform_row_major(i, j, N, Amax, Amin, snake=False):
    """
    Assign amplitudes uniformly from Amax (top-left) to Amin (bottom-right).

    If snake=True, odd rows are traversed right-to-left.
    """
    if snake and (j % 2 == 1):
        i_eff = (N - 1) - i
    else:
        i_eff = i

    idx = j * N + i_eff
    denom = N * N - 1
    t = idx / denom if denom > 0 else 0.0

    return (1 - t) * Amax + t * Amin


# =============================================================================
# Safety check
# =============================================================================

if R_um + A_min_um <= 0:
    raise ValueError("Choose R_um and A_min_um such that R_um + A_min_um > 0.")


# =============================================================================
# Build array centered about the origin
# =============================================================================

x0 = -0.5 * (N - 1) * pitchX_um
y0 = -0.5 * (N - 1) * pitchY_um

print(f"Array size: {N} x {N}")
print(f"Base radius: {R_um} µm")
print(f"Amplitude range: {A_max_um} µm to {A_min_um} µm")
print(f"Pitch: {pitchX_um} µm x {pitchY_um} µm")
print(f"Sigma scale: {sigma_scale}")
print(f"Snake mode: {snake}")
print(f"Export mode: {EXPORT_MODE}")
print(f"GDS filename: {GDS_FILENAME}")

for j in range(N):
    for i in range(N):
        A = amp_uniform_row_major(i, j, N, A_max_um, A_min_um, snake=snake)
        sigma_deg = sigma_scale * abs(A)   # sigma proportional to |A|

        xc = x0 + i * pitchX_um
        yc = y0 + j * pitchY_um

        circle_gauss_bump(
            xc, yc,
            R_um,
            A,
            theta0_deg=theta0_deg,
            sigma_deg=sigma_deg,
            n=N_samp,
            layer=layer,
        )


# =============================================================================
# Export / preview
# =============================================================================

mode = EXPORT_MODE.lower()

if mode == "gds":
    nd.export_gds(filename=GDS_FILENAME)
    print(f"GDS exported: {GDS_FILENAME}")

elif mode == "plot":
    nd.export_plt()
    print("Plot exported.")

else:
    raise ValueError("EXPORT_MODE must be one of: 'plot' or 'gds'")

Array size: 4 x 4
Base radius: 50.0 µm
Amplitude range: 49.0 µm to -49.0 µm
Pitch: 150.0 µm x 150.0 µm
Sigma scale: 0.2
Snake mode: False
Export mode: gds
GDS filename: gaussian_bump_array_4x4_R50um_A49to-49_sigmaScale0.2_20260317.gds
Starting layout export...
...gds generation
...Wrote file './gaussian_bump_array_4x4_R50um_A49to-49_sigmaScale0.2_20260317.gds'


GDS exported: gaussian_bump_array_4x4_R50um_A49to-49_sigmaScale0.2_20260317.gds
